# 19｜不用 `nn.RNN/LSTM/GRU`：从门公式实现循环网络

这不是“调一个循环层 API”的演示，而是一份可以逐行审计的最小工程实现。我们会手写 `CustomRNNCell`、`CustomLSTMCell`、`CustomGRUCell` 和变长序列扫描器，并检查参数量、张量形状、mask、梯度、受控过拟合、状态化推理与模型指纹。

> 边界：实验使用 CPU 与固定的小型合成数据，证明的是实现能够学习且合同成立，不代表真实语料上的泛化效果。

## 1. 验收目标与 API 合同

- 输入统一为 `x: [B,T,D]`，长度为 `lengths: [B]`，且每条长度必须位于 `[1,T]`。
- cell 输入一个时间步 `x_t: [B,D]`；RNN/GRU 状态是 `[B,H]`，LSTM 状态是 `(h,c)`。
- 扫描器在 padding 位置**不得更新状态**，输出 padding 位置归零。
- 分类器只消费最后一个有效状态，避免把填充值当作特征。
- 所有随机源固定；训练仅做“固定小集合受控过拟合”。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import hashlib  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 20260728  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

assert torch.__version__  # 用受控断言验证关键不变量。
assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})  # 执行当前语句以推进本节示例。

## 2. Vanilla RNN：最短的循环记忆

对时间步 $t$：

$$h_t=\tanh(W_xx_t+W_hh_{t-1}+b).$$

若输入维度为 $D$、隐藏维度为 $H$，这里把输入偏置放在 `x2h`，因此参数量是 $H(D+H+1)$。`forward` 只描述单步；时间展开稍后由扫描器显式完成。

In [ ]:
class CustomRNNCell(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_size: int, hidden_size: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if input_size <= 0 or hidden_size <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("input_size 和 hidden_size 必须为正")  # 遇到非法合同立即显式失败。
        self.input_size, self.hidden_size = input_size, hidden_size  # 计算并保存当前步骤的中间状态。
        self.x2h = nn.Linear(input_size, hidden_size, bias=True)  # 计算并保存当前步骤的中间状态。
        self.h2h = nn.Linear(hidden_size, hidden_size, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x_t: torch.Tensor, h_prev: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if x_t.ndim != 2 or h_prev.ndim != 2:  # 按当前条件选择后续控制路径。
            raise ValueError("单步输入和状态都必须是二维张量")  # 遇到非法合同立即显式失败。
        if x_t.shape[0] != h_prev.shape[0]:  # 按当前条件选择后续控制路径。
            raise ValueError("batch 维不一致")  # 遇到非法合同立即显式失败。
        return torch.tanh(self.x2h(x_t) + self.h2h(h_prev))  # 返回当前分支计算出的结果。

rnn_cell = CustomRNNCell(3, 5)  # 计算并保存当前步骤的中间状态。
sample_h = rnn_cell(torch.zeros(2, 3), torch.zeros(2, 5))  # 计算并保存当前步骤的中间状态。
assert sample_h.shape == (2, 5)  # 用受控断言验证关键不变量。
assert torch.isfinite(sample_h).all()  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in rnn_cell.parameters()) == 5 * (3 + 5 + 1)  # 用受控断言验证关键不变量。

## 3. LSTM：输入、遗忘、候选与输出门

令 $u_t=[x_t;h_{t-1}]$，一次仿射变换得到四组值：

$$i_t=\sigma(W_i u_t+b_i),\quad f_t=\sigma(W_f u_t+b_f),$$
$$g_t=\tanh(W_g u_t+b_g),\quad o_t=\sigma(W_o u_t+b_o),$$
$$c_t=f_t\odot c_{t-1}+i_t\odot g_t,\quad h_t=o_t\odot\tanh(c_t).$$

拼接实现只是计算优化，不改变公式；参数量为 $4H(D+H+1)$。门顺序在类中固定为 `i,f,g,o`，这是 checkpoint 合同的一部分。

In [ ]:
class CustomLSTMCell(nn.Module):  # 定义承载本节状态与行为的数据结构。
    gate_order = ("input", "forget", "candidate", "output")  # 计算并保存当前步骤的中间状态。

    def __init__(self, input_size: int, hidden_size: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if input_size <= 0 or hidden_size <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("维度必须为正")  # 遇到非法合同立即显式失败。
        self.input_size, self.hidden_size = input_size, hidden_size  # 计算并保存当前步骤的中间状态。
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)  # 计算并保存当前步骤的中间状态。

    def forward(self, x_t, state):  # 定义本节可复用的核心函数。
        h_prev, c_prev = state  # 计算并保存当前步骤的中间状态。
        if x_t.ndim != 2 or h_prev.shape != c_prev.shape:  # 按当前条件选择后续控制路径。
            raise ValueError("LSTM 状态必须是同形状的二维 (h,c)")  # 遇到非法合同立即显式失败。
        packed = self.gates(torch.cat([x_t, h_prev], dim=-1))  # 计算并保存当前步骤的中间状态。
        i_raw, f_raw, g_raw, o_raw = packed.chunk(4, dim=-1)  # 计算并保存当前步骤的中间状态。
        i, f, o = torch.sigmoid(i_raw), torch.sigmoid(f_raw), torch.sigmoid(o_raw)  # 计算并保存当前步骤的中间状态。
        g = torch.tanh(g_raw)  # 计算并保存当前步骤的中间状态。
        c = f * c_prev + i * g  # 计算并保存当前步骤的中间状态。
        h = o * torch.tanh(c)  # 计算并保存当前步骤的中间状态。
        return h, c  # 返回当前分支计算出的结果。

lstm_cell = CustomLSTMCell(3, 5)  # 计算并保存当前步骤的中间状态。
h1, c1 = lstm_cell(torch.randn(2, 3), (torch.zeros(2, 5), torch.zeros(2, 5)))  # 计算并保存当前步骤的中间状态。
assert h1.shape == c1.shape == (2, 5)  # 用受控断言验证关键不变量。
assert CustomLSTMCell.gate_order == ("input", "forget", "candidate", "output")  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in lstm_cell.parameters()) == 4 * 5 * (3 + 5 + 1)  # 用受控断言验证关键不变量。

## 4. GRU：两扇门合并状态

本实现采用常见的 reset-after 形式：

$$z_t=\sigma(W_zx_t+U_zh_{t-1}),\quad r_t=\sigma(W_rx_t+U_rh_{t-1}),$$
$$n_t=\tanh(W_nx_t+r_t\odot U_nh_{t-1}),$$
$$h_t=(1-z_t)\odot n_t+z_t\odot h_{t-1}.$$

不同框架可能把 reset 放在矩阵乘法之前，迁移权重时必须记录变体。本实现的参数量为 $3H(D+H+1)$。

In [ ]:
class CustomGRUCell(nn.Module):  # 定义承载本节状态与行为的数据结构。
    variant = "reset_after"  # 计算并保存当前步骤的中间状态。

    def __init__(self, input_size, hidden_size):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.input_size, self.hidden_size = input_size, hidden_size  # 计算并保存当前步骤的中间状态。
        self.x_proj = nn.Linear(input_size, 3 * hidden_size, bias=True)  # 计算并保存当前步骤的中间状态。
        self.h_proj = nn.Linear(hidden_size, 3 * hidden_size, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x_t, h_prev):  # 定义本节可复用的核心函数。
        if x_t.ndim != 2 or h_prev.ndim != 2:  # 按当前条件选择后续控制路径。
            raise ValueError("GRU 单步张量必须二维")  # 遇到非法合同立即显式失败。
        x_z, x_r, x_n = self.x_proj(x_t).chunk(3, dim=-1)  # 计算并保存当前步骤的中间状态。
        h_z, h_r, h_n = self.h_proj(h_prev).chunk(3, dim=-1)  # 计算并保存当前步骤的中间状态。
        z = torch.sigmoid(x_z + h_z)  # 计算并保存当前步骤的中间状态。
        r = torch.sigmoid(x_r + h_r)  # 计算并保存当前步骤的中间状态。
        n = torch.tanh(x_n + r * h_n)  # 计算并保存当前步骤的中间状态。
        return (1.0 - z) * n + z * h_prev  # 返回当前分支计算出的结果。

gru_cell = CustomGRUCell(3, 5)  # 计算并保存当前步骤的中间状态。
gru_h = gru_cell(torch.randn(2, 3), torch.zeros(2, 5))  # 计算并保存当前步骤的中间状态。
assert gru_h.shape == (2, 5)  # 用受控断言验证关键不变量。
assert CustomGRUCell.variant == "reset_after"  # 用受控断言验证关键不变量。
assert sum(p.numel() for p in gru_cell.parameters()) == 3 * 5 * (3 + 5 + 1)  # 用受控断言验证关键不变量。

## 5. 显式时间扫描与变长 mask

`lengths > t` 生成 `[B,1]` 活跃掩码。新状态只写入活跃样本，已结束样本保留最后有效状态；逐步输出则把 padding 位置清零。这两个动作不能混为一谈：前者保证最终表示正确，后者避免下游误读 padding。

循环在 Python 层实现，便于教学和审计；生产中长序列应替换为融合 kernel 或经过验证的官方实现。

In [ ]:
CELL_TYPES = {"rnn": CustomRNNCell, "lstm": CustomLSTMCell, "gru": CustomGRUCell}  # 计算并保存当前步骤的中间状态。

class SequenceEncoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_size, hidden_size, kind="lstm"):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if kind not in CELL_TYPES:  # 按当前条件选择后续控制路径。
            raise ValueError(f"未知 cell: {kind}")  # 遇到非法合同立即显式失败。
        self.kind, self.hidden_size = kind, hidden_size  # 计算并保存当前步骤的中间状态。
        self.cell = CELL_TYPES[kind](input_size, hidden_size)  # 计算并保存当前步骤的中间状态。

    def initial_state(self, batch, *, device, dtype):  # 定义本节可复用的核心函数。
        z = torch.zeros(batch, self.hidden_size, device=device, dtype=dtype)  # 计算并保存当前步骤的中间状态。
        return (z, z.clone()) if self.kind == "lstm" else z  # 返回当前分支计算出的结果。

    def forward(self, x, lengths, state=None):  # 定义本节可复用的核心函数。
        if x.ndim != 3 or lengths.shape != (x.shape[0],):  # 按当前条件选择后续控制路径。
            raise ValueError("期望 x=[B,T,D], lengths=[B]")  # 遇到非法合同立即显式失败。
        B, T, _ = x.shape  # 计算并保存当前步骤的中间状态。
        if bool(((lengths < 1) | (lengths > T)).any()):  # 按当前条件选择后续控制路径。
            raise ValueError("lengths 必须位于 [1,T]")  # 遇到非法合同立即显式失败。
        state = self.initial_state(B, device=x.device, dtype=x.dtype) if state is None else state  # 计算并保存当前步骤的中间状态。
        outputs = []  # 计算并保存当前步骤的中间状态。
        for t in range(T):  # 遍历输入元素以累积或检查结果。
            active = (lengths > t).unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
            if self.kind == "lstm":  # 按当前条件选择后续控制路径。
                old_h, old_c = state  # 计算并保存当前步骤的中间状态。
                new_h, new_c = self.cell(x[:, t], state)  # 计算并保存当前步骤的中间状态。
                state = (torch.where(active, new_h, old_h), torch.where(active, new_c, old_c))  # 计算并保存当前步骤的中间状态。
                out_t = torch.where(active, state[0], torch.zeros_like(state[0]))  # 计算并保存当前步骤的中间状态。
            else:  # 处理前置条件不成立的分支。
                new_h = self.cell(x[:, t], state)  # 计算并保存当前步骤的中间状态。
                state = torch.where(active, new_h, state)  # 计算并保存当前步骤的中间状态。
                out_t = torch.where(active, state, torch.zeros_like(state))  # 计算并保存当前步骤的中间状态。
            outputs.append(out_t)  # 执行当前语句以推进本节示例。
        return torch.stack(outputs, dim=1), state  # 返回当前分支计算出的结果。

class SequenceClassifier(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, input_size, hidden_size, num_classes, kind="lstm"):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.encoder = SequenceEncoder(input_size, hidden_size, kind)  # 计算并保存当前步骤的中间状态。
        self.head = nn.Linear(hidden_size, num_classes)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, lengths):  # 定义本节可复用的核心函数。
        sequence, state = self.encoder(x, lengths)  # 计算并保存当前步骤的中间状态。
        final_h = state[0] if self.encoder.kind == "lstm" else state  # 计算并保存当前步骤的中间状态。
        return self.head(final_h), sequence  # 返回当前分支计算出的结果。

for kind in CELL_TYPES:  # 遍历输入元素以累积或检查结果。
    enc = SequenceEncoder(3, 7, kind)  # 计算并保存当前步骤的中间状态。
    seq, state = enc(torch.randn(4, 6, 3), torch.tensor([6, 4, 2, 1]))  # 计算并保存当前步骤的中间状态。
    assert seq.shape == (4, 6, 7)  # 用受控断言验证关键不变量。
    assert torch.count_nonzero(seq[3, 1:]) == 0  # 用受控断言验证关键不变量。

## 6. 可诊断的合成任务

每个样本第一维的有效时间步平均值决定二分类标签；padding 故意填入大噪声。模型若正确使用长度就只能从有效片段学习。我们只在固定的 40 条样本上训练并验收，明确称为**受控过拟合**，不报告“测试集泛化”。

In [ ]:
def make_toy_batch(n=40, max_len=7, dim=3):  # 定义本节可复用的核心函数。
    g = torch.Generator().manual_seed(SEED + 1)  # 计算并保存当前步骤的中间状态。
    lengths = torch.randint(2, max_len + 1, (n,), generator=g)  # 计算并保存当前步骤的中间状态。
    x = torch.randn(n, max_len, dim, generator=g)  # 计算并保存当前步骤的中间状态。
    valid = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)  # 计算并保存当前步骤的中间状态。
    signal = (x[:, :, 0] * valid).sum(1) / lengths  # 计算并保存当前步骤的中间状态。
    y = (signal > 0).long()  # 计算并保存当前步骤的中间状态。
    # padding 的第一维加巨大干扰；正确 mask 后不应影响标签。
    pad = ~valid  # 计算并保存当前步骤的中间状态。
    x[:, :, 0] = torch.where(pad, 25.0 * torch.randn(n, max_len, generator=g), x[:, :, 0])  # 计算并保存当前步骤的中间状态。
    return x, lengths, y, valid  # 返回当前分支计算出的结果。

x_train, len_train, y_train, valid_train = make_toy_batch()  # 计算并保存当前步骤的中间状态。
assert x_train.shape == (40, 7, 3)  # 用受控断言验证关键不变量。
assert len_train.min().item() >= 2  # 用受控断言验证关键不变量。
assert set(y_train.tolist()) == {0, 1}  # 用受控断言验证关键不变量。
assert (~valid_train).any()  # 用受控断言验证关键不变量。
print({"samples": len(y_train), "positive_rate": y_train.float().mean().item()})  # 执行当前语句以推进本节示例。

## 7. 优化器、损失与梯度合同

使用交叉熵和 Adam。每步在 `backward` 后先检查梯度有限且非零，再用全局范数裁剪限制爆炸梯度。小样本循环网络的曲线会波动，因此验收同时看初末损失与训练集准确率。

In [ ]:
torch.manual_seed(SEED + 2)  # 执行当前语句以推进本节示例。
model19 = SequenceClassifier(3, 14, 2, kind="lstm").to(DEVICE)  # 计算并保存当前步骤的中间状态。
optimizer19 = torch.optim.Adam(model19.parameters(), lr=0.035)  # 计算并保存当前步骤的中间状态。
losses19, raw_grad_norms = [], []  # 计算并保存当前步骤的中间状态。

model19.train()  # 执行当前语句以推进本节示例。
for step in range(180):  # 遍历输入元素以累积或检查结果。
    optimizer19.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    logits, _ = model19(x_train, len_train)  # 计算并保存当前步骤的中间状态。
    loss = F.cross_entropy(logits, y_train)  # 计算并保存当前步骤的中间状态。
    loss.backward()  # 执行当前语句以推进本节示例。
    grads = [p.grad for p in model19.parameters() if p.grad is not None]  # 计算并保存当前步骤的中间状态。
    if step == 0:  # 按当前条件选择后续控制路径。
        assert grads and all(torch.isfinite(g).all() for g in grads)  # 用受控断言验证关键不变量。
        assert sum(float(g.abs().sum()) for g in grads) > 0  # 用受控断言验证关键不变量。
    raw_norm = torch.nn.utils.clip_grad_norm_(model19.parameters(), max_norm=1.0)  # 计算并保存当前步骤的中间状态。
    optimizer19.step()  # 执行当前语句以推进本节示例。
    losses19.append(float(loss.detach()))  # 执行当前语句以推进本节示例。
    raw_grad_norms.append(float(raw_norm))  # 执行当前语句以推进本节示例。

model19.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    train_logits, train_sequence = model19(x_train, len_train)  # 计算并保存当前步骤的中间状态。
    train_pred = train_logits.argmax(-1)  # 计算并保存当前步骤的中间状态。
    train_acc = (train_pred == y_train).float().mean().item()  # 计算并保存当前步骤的中间状态。

assert losses19[-1] < losses19[0] * 0.25  # 用受控断言验证关键不变量。
assert train_acc >= 0.95  # 用受控断言验证关键不变量。
assert all(math.isfinite(v) for v in raw_grad_norms)  # 用受控断言验证关键不变量。
assert torch.count_nonzero(train_sequence[~valid_train]) == 0  # 用受控断言验证关键不变量。
print({"loss_first": losses19[0], "loss_last": losses19[-1], "controlled_train_acc": train_acc})  # 执行当前语句以推进本节示例。

## 8. 失败反例：用 `x != 0` 猜 padding

真实特征完全可能为零，而 padding 也可能是非零占位。由内容反推 mask 会让相同有效序列因填充值不同而得到不同表示。可靠边界必须显式传 `lengths` 或上游给出的布尔 mask。

In [ ]:
probe = x_train[:3].clone()  # 计算并保存当前步骤的中间状态。
probe_changed = probe.clone()  # 计算并保存当前步骤的中间状态。
for b, length in enumerate(len_train[:3].tolist()):  # 遍历输入元素以累积或检查结果。
    probe_changed[b, length:] = 999.0  # 计算并保存当前步骤的中间状态。

with torch.no_grad():  # 在受管理的上下文中执行操作。
    good_a, _ = model19(probe, len_train[:3])  # 计算并保存当前步骤的中间状态。
    good_b, _ = model19(probe_changed, len_train[:3])  # 计算并保存当前步骤的中间状态。
    # 错误做法：谎报所有位置有效，padding 就进入状态。
    wrong_lengths = torch.full_like(len_train[:3], probe.shape[1])  # 计算并保存当前步骤的中间状态。
    bad_a, _ = model19(probe, wrong_lengths)  # 计算并保存当前步骤的中间状态。
    bad_b, _ = model19(probe_changed, wrong_lengths)  # 计算并保存当前步骤的中间状态。

assert torch.allclose(good_a, good_b, atol=1e-6)  # 用受控断言验证关键不变量。
assert not torch.allclose(bad_a, bad_b, atol=1e-3)  # 用受控断言验证关键不变量。
assert (bad_a - bad_b).abs().max().item() > 0.01  # 用受控断言验证关键不变量。
print("显式长度保持输出不变；错误长度会泄漏 padding。")  # 执行当前语句以推进本节示例。

## 9. 状态化推理与分块等价性

流式推理可以把上一块的状态传入下一块，但必须明确会话边界、batch 重排规则和状态过期时间。下方验证：在无 padding 的同一序列上，“整段扫描”的末状态与“两块扫描”的末状态一致。生产系统若跨用户复用状态，会造成严重数据串线。

In [ ]:
stream_encoder = model19.encoder  # 计算并保存当前步骤的中间状态。
stream_x = torch.randn(2, 6, 3, generator=torch.Generator().manual_seed(SEED + 3))  # 计算并保存当前步骤的中间状态。
full_lengths = torch.tensor([6, 6])  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    _, full_state = stream_encoder(stream_x, full_lengths)  # 计算并保存当前步骤的中间状态。
    _, state_a = stream_encoder(stream_x[:, :2], torch.tensor([2, 2]))  # 计算并保存当前步骤的中间状态。
    _, state_b = stream_encoder(stream_x[:, 2:], torch.tensor([4, 4]), state=state_a)  # 计算并保存当前步骤的中间状态。

assert torch.allclose(full_state[0], state_b[0], atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(full_state[1], state_b[1], atol=1e-6)  # 用受控断言验证关键不变量。
assert full_state[0].shape == (2, 14)  # 用受控断言验证关键不变量。

## 10. 保存合同：配置、权重指纹与可重复加载

仅保存 `state_dict` 不足以解释权重：还要保存 cell 类型、门顺序、维度、随机种子、PyTorch 版本和数据/标签协议。这里用内存缓冲区模拟制品，SHA-256 指纹可用于部署前校验字节是否被替换。

In [ ]:
manifest19 = {  # 计算并保存当前步骤的中间状态。
    "artifact": "sequence_classifier",  # 执行当前语句以推进本节示例。
    "schema_version": 1,  # 执行当前语句以推进本节示例。
    "cell": "custom_lstm_ifgo",  # 执行当前语句以推进本节示例。
    "input_size": 3,  # 执行当前语句以推进本节示例。
    "hidden_size": 14,  # 执行当前语句以推进本节示例。
    "classes": ["mean_non_positive", "mean_positive"],  # 执行当前语句以推进本节示例。
    "seed": SEED,  # 执行当前语句以推进本节示例。
    "torch_version": torch.__version__,  # 执行当前语句以推进本节示例。
    "mask_contract": "lengths in [1,T], padding state frozen",  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
buffer19 = io.BytesIO()  # 计算并保存当前步骤的中间状态。
torch.save({"manifest": manifest19, "state_dict": model19.state_dict()}, buffer19)  # 执行当前语句以推进本节示例。
payload19 = buffer19.getvalue()  # 计算并保存当前步骤的中间状态。
fingerprint19 = hashlib.sha256(payload19).hexdigest()  # 计算并保存当前步骤的中间状态。

buffer19.seek(0)  # 执行当前语句以推进本节示例。
loaded19 = torch.load(buffer19, map_location="cpu", weights_only=False)  # 计算并保存当前步骤的中间状态。
clone19 = SequenceClassifier(3, 14, 2, kind="lstm")  # 计算并保存当前步骤的中间状态。
clone19.load_state_dict(loaded19["state_dict"])  # 执行当前语句以推进本节示例。
clone19.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    clone_logits, _ = clone19(x_train, len_train)  # 计算并保存当前步骤的中间状态。

assert loaded19["manifest"]["schema_version"] == 1  # 用受控断言验证关键不变量。
assert len(fingerprint19) == 64  # 用受控断言验证关键不变量。
assert torch.allclose(train_logits, clone_logits, atol=1e-7)  # 用受控断言验证关键不变量。
assert set(loaded19["state_dict"]) == set(model19.state_dict())  # 用受控断言验证关键不变量。
print({"sha256": fingerprint19[:16] + "…", "bytes": len(payload19)})  # 执行当前语句以推进本节示例。

## 11. 生产观测与安全边界

上线至少记录：输入长度分布、超长截断率、空序列拒绝数、每批延迟、梯度范数、损失、类别分布漂移和模型指纹。状态化服务还要以租户与会话双键隔离状态，并在退出/超时后删除。

不要直接加载不可信的 pickle 权重；优先使用受控制品仓、哈希校验和 `weights_only=True` 能覆盖的格式。输入维度与长度应在进入模型前限流，否则循环长度本身就是拒绝服务面。

## 12. 何时替换教学实现

- 需要 GPU 吞吐、混合精度或长序列：换成官方融合循环层，并用本笔记的数值测试做回归。
- 需要跨框架导出：固定门顺序、GRU 变体、batch 维和初始状态语义。
- 需要真实泛化结论：按实体/用户/时间切分独立验证集与测试集，报告置信区间，而不是复用这里的过拟合准确率。
- 需要多层/双向网络：明确每层 dropout 的位置以及双向状态拼接顺序。

## 13. 原始资料与实现依据

- Elman, *Finding Structure in Time* (1990)：https://doi.org/10.1016/0364-0213(90)90002-E
- Hochreiter & Schmidhuber, *Long Short-Term Memory* (1997)：https://doi.org/10.1162/neco.1997.9.8.1735
- Cho et al., *Learning Phrase Representations using RNN Encoder–Decoder* (2014)：https://arxiv.org/abs/1406.1078
- PyTorch `nn.Module` 官方文档：https://pytorch.org/docs/stable/generated/torch.nn.Module.html
- PyTorch 梯度裁剪官方文档：https://pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html

公式来自论文；代码是为可解释性重新实现的教学版本。

## 14. 面试式复盘

1. LSTM 的 cell state 为何比纯 RNN 更容易保留梯度？关键是加法更新路径与门控。
2. mask 为什么既要冻结状态又要清零逐步输出？两者保护不同下游接口。
3. “训练准确率 100%”为什么不是好模型证据？这里只验证代码可学习，没有独立分布的泛化证明。
4. checkpoint 迁移最容易漏什么？门顺序、GRU 变体、标签映射、预处理和版本。
5. 流式状态为何属于安全问题？状态串会话会直接泄露另一用户的上下文。

In [ ]:
# 最终合同测试：覆盖拒绝非法长度、确定性推理与 cell 的梯度路径。
try:  # 尝试执行可能失败的受控操作。
    model19(torch.randn(2, 3, 3), torch.tensor([3, 0]))  # 执行当前语句以推进本节示例。
    invalid_length_rejected = False  # 计算并保存当前步骤的中间状态。
except ValueError:  # 捕获预期异常并验证失败分支。
    invalid_length_rejected = True  # 计算并保存当前步骤的中间状态。

for kind in CELL_TYPES:  # 遍历输入元素以累积或检查结果。
    gradient_probe = SequenceEncoder(3, 4, kind)  # 计算并保存当前步骤的中间状态。
    probe_sequence, _ = gradient_probe(torch.randn(2, 3, 3), torch.tensor([3, 2]))  # 计算并保存当前步骤的中间状态。
    probe_sequence.square().mean().backward()  # 执行当前语句以推进本节示例。
    probe_grads = [p.grad for p in gradient_probe.parameters()]  # 计算并保存当前步骤的中间状态。
    assert all(g is not None and torch.isfinite(g).all() for g in probe_grads)  # 用受控断言验证关键不变量。
    assert sum(float(g.abs().sum()) for g in probe_grads) > 0  # 用受控断言验证关键不变量。

with torch.no_grad():  # 在受管理的上下文中执行操作。
    repeat_logits, _ = model19(x_train, len_train)  # 计算并保存当前步骤的中间状态。

assert invalid_length_rejected  # 用受控断言验证关键不变量。
assert torch.allclose(train_logits, repeat_logits)  # 用受控断言验证关键不变量。
assert manifest19["cell"] == "custom_lstm_ifgo"  # 用受控断言验证关键不变量。
assert manifest19["mask_contract"].startswith("lengths")  # 用受控断言验证关键不变量。
assert losses19[-1] < 0.2  # 用受控断言验证关键不变量。
assert 0.0 <= train_acc <= 1.0  # 用受控断言验证关键不变量。
assert fingerprint19 == hashlib.sha256(payload19).hexdigest()  # 用受控断言验证关键不变量。
print("Notebook 19：全部合同测试通过。")  # 执行当前语句以推进本节示例。